# optimize_check_data

Rustシミュレーションの `*_pop.arrow` を **Polars** で確認し、`calc_score()` の考え方が妥当かを検証するためのノートです。

このノートで確認すること:
1. `pop.arrow` の列名と構造
2. `num_iter` が何を表していそうか
3. 各 `num_iter` の最終行を取る現在の `calc_score()` が妥当か


## 事前準備

必要なら先に以下を実行してください。

```bash
pip install polars pyarrow
```

※ `pyarrow` は Arrow/IPC 系ファイル読み取りの依存として必要になる場合があります。


In [1]:
from pathlib import Path
import polars as pl

In [2]:
# ===== ここを必要に応じて変更 =====
RESULT_DIR = Path("temp_test_2_experiment/result")
IDENTIFIER = "trial_best"   # 例: "trial_0", "trial_1", "trial_best"
TOTAL_AGENTS = 1000

arrow_path = RESULT_DIR / f"{IDENTIFIER}_pop.arrow"
arrow_path


PosixPath('temp_test_2_experiment/result/trial_best_pop.arrow')

In [3]:
if not arrow_path.exists():
    raise FileNotFoundError(f"ファイルが見つかりません: {arrow_path.resolve()}")

df = pl.read_ipc(arrow_path)
df


num_iter,t,num_selfish
u32,u32,u32
7,0,1
7,1,4
7,2,13
7,3,47
7,4,93
…,…,…
77,10,7
77,11,1
77,12,0


In [4]:
print("shape =", df.shape)
print("columns =", df.columns)
print("\nschema =")
for k, v in df.schema.items():
    print(f"  {k}: {v}")


shape = (1228, 3)
columns = ['num_iter', 't', 'num_selfish']

schema =
  num_iter: UInt32
  t: UInt32
  num_selfish: UInt32


In [5]:
print("head:")
display(df.head(10))

print("tail:")
display(df.tail(10))


head:


num_iter,t,num_selfish
u32,u32,u32
7,0,1
7,1,4
7,2,13
7,3,47
7,4,93
7,5,47
7,6,16
7,7,3
7,8,0


tail:


num_iter,t,num_selfish
u32,u32,u32
77,5,25
77,6,16
77,7,31
77,8,47
77,9,22
77,10,7
77,11,1
77,12,0
77,13,0


## `num_iter` の確認

`calc_score()` が妥当かどうかを判断するには、`num_iter` が
- モンテカルロ反復番号なのか
- 時刻ステップなのか

を見極める必要があります。


In [6]:
if "num_iter" in df.columns:
    summary = (
        df.group_by("num_iter")
          .len()
          .sort("num_iter")
    )
    display(summary)
else:
    print("num_iter 列はありません。")


num_iter,len
u32,u32
0,9
1,15
2,10
3,10
4,11
…,…
95,11
96,11
97,12


In [7]:
# 時間・ステップに相当しそうな列を探す
candidate_time_cols = [c for c in df.columns if c.lower() in {"step", "num_step", "time", "t"}]
candidate_time_cols


['t']

In [8]:
# num_iter がある場合、その中身を少し見る
if "num_iter" in df.columns:
    first_iter = df.select(pl.col("num_iter").min()).item()
    print("first num_iter =", first_iter)
    display(df.filter(pl.col("num_iter") == first_iter).head(20))
else:
    print("num_iter 列がないためスキップしました。")


first num_iter = 0


num_iter,t,num_selfish
u32,u32,u32
0,0,0
0,1,3
0,2,48
0,3,114
0,4,70
0,5,20
0,6,2
0,7,0
0,8,0


## 現在の `calc_score()` 相当の計算

現行コードは概ね以下に相当します。

- `num_iter` がなければ最後の `num_selfish`
- `num_iter` があれば各 `num_iter` の末尾の `num_selfish` を平均


In [12]:
def calc_score_current_polars(df: pl.DataFrame, total_agents: int = 1000) -> float:
    if "num_selfish" not in df.columns:
        raise ValueError("num_selfish 列がありません。")

    if "num_iter" not in df.columns:
        return float(df["num_selfish"][-1]) / total_agents

    final_rows = (
        df.with_row_index("row_idx")
          .group_by("num_iter")
          .agg(pl.all().sort_by("row_idx").last())
    )
    avg_selfish = final_rows["num_selfish"].mean()
    return float(avg_selfish) / total_agents

score_current = calc_score_current_polars(df, TOTAL_AGENTS)
print("current-style score =", score_current)


current-style score = 1e-05


## より明示的な「最終ステップ」計算

もし時刻列があるなら、各 `num_iter` で最終時刻の行を明示的に取る版も比較します。


In [13]:
def calc_score_explicit_final(df: pl.DataFrame, total_agents: int = 1000):
    if "num_selfish" not in df.columns:
        raise ValueError("num_selfish 列がありません。")

    if "num_iter" not in df.columns:
        return float(df["num_selfish"][-1]) / total_agents

    time_col = None
    for c in ["step", "num_step", "time", "t"]:
        if c in df.columns:
            time_col = c
            break

    if time_col is None:
        return None

    final_rows = (
        df.group_by("num_iter")
          .agg(pl.all().sort_by(time_col).last())
    )
    return float(final_rows["num_selfish"].mean()) / total_agents

score_explicit = calc_score_explicit_final(df, TOTAL_AGENTS)
print("explicit-final score =", score_explicit)


explicit-final score = 1e-05


## 判断の目安

- `num_iter` ごとに複数行あり、さらに各行が時系列なら、現行 `calc_score()` はかなり妥当です。
- `num_iter` が無いなら、単発結果の最後の行を使う今の実装でよい可能性が高いです。
- 時刻列があり、`current-style score` と `explicit-final score` がズレるなら、`calc_score()` の修正候補です。


In [11]:
def calc_score(identifier: str) -> float:
    arrow_path = RESULT_DIR / f"{identifier}_pop.arrow"
    if not arrow_path.exists():
        return 1.0

    try:
        df = pd.read_feather(arrow_path)
        if "num_selfish" not in df.columns:
            return 1.0

        if "num_iter" in df.columns and "t" in df.columns:
            final_rows = (
                df.sort_values(["num_iter", "t"])
                  .groupby("num_iter", as_index=False)
                  .tail(1)
            )
            avg_selfish = final_rows["num_selfish"].mean()
        elif "num_iter" in df.columns:
            avg_selfish = df.groupby("num_iter")["num_selfish"].last().mean()
        else:
            avg_selfish = df["num_selfish"].iloc[-1]

        return avg_selfish / TOTAL_AGENTS

    except Exception as e:
        print(f"  [Error] Score calculation failed: {e}", file=sys.stderr)
        return 1.0